# Colab VLM 서버 (ngrok) — `scripts/analyze_top_places.py`용 HF_TOKEN 우회 경로

`docs/adr/0003-place-image-vlm-analysis.md`가 기본으로 삼는 경로는 Hugging Face
**Inference API**(`HF_TOKEN` 필요)다. 이 노트북은 그 대신 **Colab 무료 GPU에서 공개
VLM(Qwen2.5-VL-7B-Instruct)을 직접 로드**해 추론하고, `ngrok`으로 노출해
**HF_TOKEN 없이** 같은 배치 스크립트를 돌릴 수 있게 한다.

## 사용 순서

1. **런타임 → 런타임 유형 변경 → T4 GPU** 선택.
2. 셀을 **위에서부터 순서대로** 실행. 마지막 셀이 `https://xxxx.ngrok-free.dev` URL을 출력한다.
3. 로컬(이 리포)에서:
   ```bash
   python scripts/analyze_top_places.py \
       --backend-base-url http://localhost:8080 \
       --hf-endpoint-url https://xxxx.ngrok-free.dev
   ```

## 세션이 끊겼다면 (필독)

런타임 연결이 끊기거나 ngrok이 `ERR_NGROK_3200 (endpoint offline)`을 반환하면 **새 런타임**이라
설치·모델 로드가 전부 사라진 상태다. 중간 셀만 재실행하지 말고 **1번 셀부터 전부 다시 실행**할 것.
배치는 매 실행마다 `place_insights.json`을 통째로 덮어쓰므로 중단 지점부터 이어지지 않는다.

## 설계 노트 — 왜 이렇게 설치하는가

Colab 이미지에는 `torch` / `torchvision` / `torchaudio`가 **CUDA 빌드까지 서로 맞춰진 세트**로
미리 깔려 있다. 여기에 `pip install -U`로 저 셋 중 하나라도 건드리면 CUDA 버전이 어긋나
`transformers`의 지연 임포트가 깨진다 (실제로 겪은 오류:
`PyTorch has CUDA version 13.0 whereas TorchAudio has CUDA version 12.8`).

그래서 이 노트북은:

- **torch/torchvision/torchaudio를 절대 설치·업그레이드·삭제하지 않는다.** Colab 기본 세트를 그대로 쓴다.
- `qwen-vl-utils[decord]`를 쓰지 않는다 — 비디오용 무거운 의존성을 끌어오는데, 우리는 정지 이미지
  1장만 다룬다. `PIL.Image`를 processor에 직접 넘기면 충분하다.
- `Pillow`도 업그레이드하지 않되, Colab 이미지에서 `PIL._typing._Ink`가 없어 `PIL.ImageDraw`
  임포트가 깨지는 경우가 있어(torchvision이 이 경로를 탄다) **타입힌트 전용 심볼이므로 없으면
  더미로 채워** 우회한다.
- 그 결과 설치하는 건 순수 파이썬 패키지뿐이라 네이티브 빌드 충돌이 날 여지가 없다.

In [ ]:
# 1. GPU 확인 — "Tesla T4" 같은 이름이 나와야 한다.
#    아무것도 안 나오면 런타임 → 런타임 유형 변경 → T4 GPU를 먼저 선택할 것.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. 의존성 설치
#    torch / torchvision / torchaudio 는 건드리지 않는다 (상단 "설계 노트" 참고).
#    아래는 전부 순수 파이썬 패키지라 CUDA 빌드 충돌이 발생하지 않는다.
!pip install -q -U "transformers>=4.49" accelerate bitsandbytes \
    fastapi uvicorn nest_asyncio pyngrok

print("설치 완료 — 다음 셀에서 임포트를 검증한다.")

In [ ]:
# 3. 환경 검증 — 문제가 되는 임포트를 실제 사용 전에 여기서 전부 트리거해 본다.
#    이 셀이 통과하면 이후 셀에서 임포트 오류가 날 일은 없다.

# Colab 이미지의 Pillow는 PIL.ImageText가 PIL._typing._Ink를 임포트하는데 그 심볼이
# 없는 경우가 있다. 타입힌트 전용이라 런타임 로직에는 쓰이지 않으므로 더미로 채운다.
# (torchvision이 PIL.ImageDraw를 임포트하면서 이 경로를 탄다.)
import PIL._typing as _pil_typing

if not hasattr(_pil_typing, "_Ink"):
    _pil_typing._Ink = object
    print("PIL._typing._Ink 없음 -> 더미로 패치")

from PIL import Image, ImageDraw  # noqa: F401  (ImageDraw = 깨지던 경로 검증용)

import torch
import torchvision
import transformers
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration  # noqa: F401

print("torch        :", torch.__version__)
print("torchvision  :", torchvision.__version__)
print("transformers :", transformers.__version__)
print("CUDA 사용가능 :", torch.cuda.is_available())

assert torch.cuda.is_available(), "GPU 런타임이 아니다 — 런타임 유형을 T4 GPU로 바꾸고 처음부터 다시 실행할 것."
print("\n환경 검증 통과.")

In [ ]:
# 4. ngrok 인증 토큰 (ngrok.com → 로그인 → 대시보드 → Your Authtoken, 무료)
from getpass import getpass

from pyngrok import conf, ngrok

NGROK_AUTHTOKEN = getpass("NGROK_AUTHTOKEN: ")
conf.get_default().auth_token = NGROK_AUTHTOKEN
ngrok.set_auth_token(NGROK_AUTHTOKEN)
print("ngrok 토큰 설정 완료.")

In [ ]:
# 5. 모델 로드 (Qwen2.5-VL-7B-Instruct, 4bit 양자화) — 다운로드 포함 몇 분 걸린다.
#    scripts/analyze_top_places.py의 DEFAULT_HF_MODEL과 동일한 모델.
#    VRAM이 부족하면 MODEL_ID를 "Qwen/Qwen2.5-VL-3B-Instruct"로 낮춘다.
import torch
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

# 입력 해상도 상한. 관광지 원본 사진은 수천 px이라 그대로 넣으면 비전 토큰이 폭증해
# T4에서 느려지거나 OOM이 난다. 장면/조명/구도 판단에는 이 정도면 충분하다.
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 1024 * 28 * 28

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS
)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()
print("모델 로드 완료:", MODEL_ID)

In [ ]:
# 6. 추론 함수 — base64 이미지 + 프롬프트 -> 모델 원문 텍스트
#    qwen_vl_utils(decord 등 비디오 의존성)를 쓰지 않고 PIL 이미지를 직접 넘긴다.
import base64
import io

import torch
from PIL import Image


def run_vlm(image_b64: str, prompt: str, max_tokens: int = 500) -> str:
    image = Image.open(io.BytesIO(base64.b64decode(image_b64))).convert("RGB")
    messages = [
        {
            "role": "user",
            "content": [{"type": "image"}, {"type": "text", "text": prompt}],
        }
    ]
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(text=[text], images=[image], return_tensors="pt").to(model.device)

    with torch.inference_mode():
        generated = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)

    # 프롬프트 부분을 잘라내고 새로 생성된 토큰만 디코드한다.
    trimmed = generated[0][inputs.input_ids.shape[1] :]
    return processor.decode(trimmed, skip_special_tokens=True)


print("추론 함수 준비 완료.")

In [ ]:
# 7. FastAPI 서버
#    계약: scripts/analyze_top_places.py::call_custom_endpoint_with_retry 가 기대하는
#    POST /analyze {image_b64, mime, prompt, max_tokens} -> {"text": "..."}
import logging

from fastapi import FastAPI
from pydantic import BaseModel

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("vlm-server")

app = FastAPI()


class AnalyzeRequest(BaseModel):
    image_b64: str
    mime: str = "image/jpeg"
    prompt: str
    max_tokens: int = 500


@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_ID}


@app.post("/analyze")
def analyze(req: AnalyzeRequest):
    # 이미지 1장이 실패해도 서버가 죽으면 배치 전체가 무너진다. 개별 실패는 빈
    # 텍스트로 돌려주고, 호출부가 그 장만 건너뛰게 한다.
    try:
        text = run_vlm(req.image_b64, req.prompt, req.max_tokens)
        return {"text": text}
    except Exception as exc:
        logger.exception("분석 실패")
        return {"text": "", "error": f"{type(exc).__name__}: {exc}"}


print("FastAPI 앱 정의 완료.")

In [ ]:
# 8. 서버 기동 + 로컬 스모크 테스트 (ngrok 노출 전에 실제 추론까지 확인한다)
import base64
import io
import threading
import time

import nest_asyncio
import requests
import uvicorn
from PIL import Image

nest_asyncio.apply()

PORT = 8000

if not globals().get("_SERVER_STARTED"):
    threading.Thread(
        target=lambda: uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning"),
        daemon=True,
    ).start()
    _SERVER_STARTED = True
    time.sleep(3)

# 단색이 아닌 실제 그림이 있는 작은 테스트 이미지 (모델이 뭐라도 말할 거리가 있어야 한다):
# 위쪽은 하늘색, 아래쪽은 모래색 — 대충 해변처럼 보이는 2색 이미지.
buf = io.BytesIO()
img = Image.new("RGB", (256, 256), (135, 180, 220))
img.paste(Image.new("RGB", (256, 80), (230, 215, 185)), (0, 176))
img.save(buf, format="JPEG")
sample_b64 = base64.b64encode(buf.getvalue()).decode()

print("health:", requests.get(f"http://localhost:{PORT}/health", timeout=30).json())

resp = requests.post(
    f"http://localhost:{PORT}/analyze",
    json={
        "image_b64": sample_b64,
        "mime": "image/jpeg",
        "prompt": "이 이미지를 한 문장으로 설명하세요.",
        "max_tokens": 64,
    },
    timeout=300,
)
print("analyze:", resp.status_code, resp.json())
assert resp.json().get("text"), "추론 결과가 비었다 — 위 로그를 확인할 것."
print("\n스모크 테스트 통과.")

In [ ]:
# 9. ngrok으로 노출 — 이 출력의 URL을 --hf-endpoint-url 로 쓴다.
from pyngrok import ngrok

for tunnel in ngrok.get_tunnels():  # 재실행 시 이전 터널 정리
    ngrok.disconnect(tunnel.public_url)

public_url = ngrok.connect(PORT, "http").public_url

print("HF_ENDPOINT_URL =", public_url)
print("\n로컬에서 이렇게 실행:\n")
print(
    f"python scripts/analyze_top_places.py \\\n"
    f"    --backend-base-url http://localhost:8080 \\\n"
    f"    --hf-endpoint-url {public_url}"
)
print("\n※ 배치가 끝날 때까지 이 탭을 활성 상태로 열어둘 것 (백그라운드 방치 시 세션이 끊긴다).")

In [ ]:
# 10. (배치가 다 끝난 뒤 실행) 터널 정리
# from pyngrok import ngrok
# ngrok.kill()
# print("ngrok 터널을 닫았다.")